In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/credit_default_clean.csv")

df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
df.columns.tolist()

['ID',
 'LIMIT_BAL',
 'SEX',
 'EDUCATION',
 'MARRIAGE',
 'AGE',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'default payment next month']

In [5]:
# Rename target
df = df.rename(columns={"default payment next month": "default"})

# Column groups
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
pay_cols = [f"PAY_AMT{i}" for i in range(1, 7)]
status_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

# Engineer features
df["AVG_BILL_AMT"] = df[bill_cols].mean(axis=1)
df["AVG_PAY_AMT"] = df[pay_cols].mean(axis=1)
df["CREDIT_UTIL"] = df["AVG_BILL_AMT"] / df["LIMIT_BAL"]
df["DELAY_COUNT"] = (df[status_cols] > 0).sum(axis=1)
df["MAX_DELAY"] = df[status_cols].max(axis=1)

In [6]:
df[["AVG_BILL_AMT", "AVG_PAY_AMT", "CREDIT_UTIL",
    "DELAY_COUNT", "MAX_DELAY"]].describe().round(2)

,AVG_BILL_AMT,AVG_PAY_AMT,CREDIT_UTIL,DELAY_COUNT,MAX_DELAY
count,30000.00,30000.00,30000.00,30000.00,30000.00
mean,44976.95,5275.23,0.37,0.83,0.44
std,63260.72,10137.95,0.35,1.55,1.35
min,-56043.17,0.00,-0.23,0.00,-2.00
25%,4781.33,1113.29,0.03,0.00,0.00
50%,21051.83,2397.17,0.28,0.00,0.00
75%,57104.42,5583.92,0.69,1.00,2.00
max,877313.83,627344.33,5.36,6.00,8.00


In [7]:
df.to_csv("../data/processed/credit_default_engineered.csv", index=False)

print(df.shape)

(30000, 30)


## Conclusion

Five features were created to summarise customer credit usage and repayment behaviour:

- **AVG_BILL_AMT** – average monthly bill
- **AVG_PAY_AMT** – average monthly payment
- **CREDIT_UTIL** – average bill relative to credit limit
- **DELAY_COUNT** – number of months with delayed payments
- **MAX_DELAY** – worst repayment delay

The engineered dataset was saved for use in model training.